# EXP-31: EXP-24 + Threshold Refinado + Guardrails Ajustados

**Competition:** SPR 2026 — Mammography Report Classification  
**Base:** EXP-24 (0.81021 LB)  
**Strategy:** Mesmo pipeline, melhor pós-processamento  

**O que muda vs EXP-24:**
1. **Threshold search 2-pass**: Pass 1 coarse (step 0.02), Pass 2 fine (step 0.002) ao redor dos melhores
2. **Threshold search com orderings alternativos**: Testa múltiplas ordens de classes
3. **Guardrails mais seletivos**: BI-RADS override só se report tem ESTRUTURA de conclusão formal
4. **Classe 2 guardrail refinado**: Adiciona regras para mama normal/exame negativo
5. **Ensemble alpha search**: Testa 0.3-0.7 no blend meta+v12

**Hipótese:** O threshold greedy sequencial pode ficar preso em ótimo local. Multiple orderings + 2-pass podem encontrar thresholds melhores.  
**Runtime:** ~1.2x EXP-24 (CPU only)

In [ ]:
import os, re, hashlib, warnings, time
from itertools import permutations
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.pipeline import FeatureUnion
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb
warnings.filterwarnings('ignore')

TRAIN_PATH = '/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv'
TEST_PATH  = '/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

N_FOLDS = 5
N_CLASSES = 7

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Class distribution:\n{train["target"].value_counts().sort_index()}')

In [ ]:
# =============================================================================
# Cell 2: Text Preprocessing (IDENTICO ao EXP-24)
# =============================================================================

def clean_achados(s):
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{2,}", "\n", s)
    match = re.search(r'achados[:\s]*(.*?)(?:impress[aã]o|conclus[aã]o|an[aá]lise comparativa|opini[aã]o|$)', s, flags=re.DOTALL)
    if match: s = match.group(1).strip()
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def clean_full(s):
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def extract_conclusion(s):
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    match = re.search(r'(?:impress[aã]o|conclus[aã]o|opini[aã]o)[:\s]*(.*?)(?:an[aá]lise comparativa|recomenda|$)', s, flags=re.DOTALL)
    if match:
        c = match.group(1).strip()
        c = re.sub(r'[0-9]+,[0-9]+', 'NUM', c)
        c = re.sub(r'[0-9]+', 'NUM', c)
        return c
    return ""

def stable_hash(s):
    return hashlib.md5(s.encode("utf-8")).hexdigest()

train["achados"]    = train["report"].apply(clean_achados)
train["full"]       = train["report"].apply(clean_full)
train["conclusion"] = train["report"].apply(extract_conclusion)
test["achados"]     = test["report"].apply(clean_achados)
test["full"]        = test["report"].apply(clean_full)
test["conclusion"]  = test["report"].apply(extract_conclusion)
train["group"]      = train["report"].apply(stable_hash)

y = train["target"].astype(int).values
groups = train["group"].values

print(f'Achados: {(train["achados"] != "").sum()}/{len(train)}, Conclusion: {(train["conclusion"] != "").sum()}/{len(train)}')

In [ ]:
# =============================================================================
# Cell 3: Dense Features (IDENTICO ao EXP-24)
# =============================================================================

def extract_birads_number(text):
    match = re.search(r'(?:bi-?rads|categoria)\s*:?\s*(\d)', text)
    return int(match.group(1)) if match else -1

def extract_dense_features(df):
    feat = pd.DataFrame(index=df.index)
    text_col = df['report'].fillna('').astype(str).str.lower()
    feat['report_length']   = text_col.apply(len)
    feat['has_measurement'] = text_col.str.contains(r'\b(?:cm|mm|medindo)\b', regex=True).astype(int)
    feat['has_spiculation'] = text_col.str.contains(r'espiculad', regex=True).astype(int)
    feat['has_distortion']  = text_col.str.contains(r'distor[cç][aã]o arquitetural', regex=True).astype(int)
    feat['has_biopsy']      = text_col.str.contains(r'biopsy|bi[oó]psia|resultado de cine|carcinoma', regex=True).astype(int)
    feat['word_count']        = text_col.str.split().str.len().fillna(0).astype(int)
    feat['line_count']        = text_col.str.count('\n') + 1
    feat['achados_length']    = df['achados'].str.len() if 'achados' in df.columns else 0
    feat['conclusion_length'] = df['conclusion'].str.len() if 'conclusion' in df.columns else 0
    feat['has_nodule']        = text_col.str.contains(r'n[oó]dulo', regex=True).astype(int)
    feat['has_calcification'] = text_col.str.contains(r'calcifica[cç]', regex=True).astype(int)
    feat['has_microcalc']     = text_col.str.contains(r'microcalcifica', regex=True).astype(int)
    feat['has_asymmetry']     = text_col.str.contains(r'assimetria', regex=True).astype(int)
    feat['has_lymphnode']     = text_col.str.contains(r'linfonodo|axilar', regex=True).astype(int)
    feat['has_suspicious']    = text_col.str.contains(r'suspeit[oa]|malign[oa]', regex=True).astype(int)
    feat['has_benign']        = text_col.str.contains(r'benign[oa]|cisto[s]?\b', regex=True).astype(int)
    feat['has_gross_calc']    = text_col.str.contains(r'calcifica[cç][aã]o grosseira|calcifica[cç][aã]o vascular', regex=True).astype(int)
    feat['has_nodulo_espic']  = text_col.str.contains(r'n[oó]dulo\s+espiculad', regex=True).astype(int)
    feat['has_heterogeneo']   = text_col.str.contains(r'heterog[eê]ne', regex=True).astype(int)
    feat['has_irregular']     = text_col.str.contains(r'irregular', regex=True).astype(int)
    feat['has_bilateral']     = text_col.str.contains(r'bilateral', regex=True).astype(int)
    feat['has_birads_explicit'] = text_col.str.contains(r'bi-?rads|categoria\s*\d', regex=True).astype(int)
    feat['birads_mentioned']    = text_col.apply(extract_birads_number)
    feat['risk_score'] = (
        feat['has_spiculation'] * 3 + feat['has_distortion'] * 3 +
        feat['has_nodulo_espic'] * 4 + feat['has_biopsy'] * 5 +
        feat['has_suspicious'] * 3 + feat['has_irregular'] * 2 +
        feat['has_microcalc'] * 2 + feat['has_asymmetry'] * 2 -
        feat['has_benign'] * 2 - feat['has_gross_calc'] * 2
    )
    return csr_matrix(feat.values.astype(np.float32))

X_train_dense = extract_dense_features(train)
X_test_dense  = extract_dense_features(test)
print(f'Dense features: {X_train_dense.shape[1]}')

In [ ]:
# =============================================================================
# Cell 4: TF-IDF + Level 1 + Level 2 (IDENTICO ao EXP-24)
# =============================================================================

print("Building TF-IDF features...")

tfidf_A = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=80000))
])
X_train_A = tfidf_A.fit_transform(train["achados"].values)
X_test_A  = tfidf_A.transform(test["achados"].values)

tfidf_F = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=80000))
])
X_train_F = tfidf_F.fit_transform(train["full"].values)
X_test_F  = tfidf_F.transform(test["full"].values)

tfidf_F2 = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 6), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=100000))
])
X_train_F2 = tfidf_F2.fit_transform(train["full"].values)
X_test_F2  = tfidf_F2.transform(test["full"].values)

tfidf_C = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)
X_train_C = tfidf_C.fit_transform(train["conclusion"].values)
X_test_C  = tfidf_C.transform(test["conclusion"].values)

X_train_lgb = hstack([X_train_A, X_train_dense]).tocsr()
X_test_lgb  = hstack([X_test_A,  X_test_dense]).tocsr()

X_train_full_dense = hstack([X_train_F, X_train_C, X_train_dense]).tocsr()
X_test_full_dense  = hstack([X_test_F,  X_test_C, X_test_dense]).tocsr()

print(f"Features built. Starting Level 1...")

def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

gkf = GroupKFold(n_splits=N_FOLDS)
folds = list(gkf.split(X_train_A, y, groups))

L1_MODELS = [
    ('svc_A', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=10000), cv=3, method='sigmoid'),
     X_train_A, X_test_A),
    ('svc_F', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=10000), cv=3, method='sigmoid'),
     X_train_F, X_test_F),
    ('svc_F2', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=10000), cv=3, method='sigmoid'),
     X_train_F2, X_test_F2),
    ('lgb1', lambda: lgb.LGBMClassifier(
        class_weight='balanced', n_estimators=300, learning_rate=0.05,
        max_depth=6, random_state=42, n_jobs=-1, verbose=-1),
     X_train_lgb, X_test_lgb),
    ('lgb2', lambda: lgb.LGBMClassifier(
        class_weight='balanced', n_estimators=500, learning_rate=0.03,
        max_depth=7, num_leaves=63, min_child_samples=20,
        subsample=0.8, colsample_bytree=0.6,
        random_state=123, n_jobs=-1, verbose=-1),
     X_train_lgb, X_test_lgb),
    ('lr', lambda: LogisticRegression(
        class_weight='balanced', C=1.0, max_iter=5000,
        solver='lbfgs', multi_class='multinomial', random_state=42, n_jobs=-1),
     X_train_full_dense, X_test_full_dense),
    ('ridge', lambda: RidgeClassifier(class_weight='balanced', alpha=1.0),
     X_train_F, X_test_F),
]

oof_probas = {}
test_probas = {}
t0 = time.time()
for name, model_fn, X_tr, X_te in L1_MODELS:
    oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
    te_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
    for fi, (tr_idx, va_idx) in enumerate(folds):
        m = model_fn()
        m.fit(X_tr[tr_idx], y[tr_idx])
        if hasattr(m, 'predict_proba'):
            oof[va_idx] = m.predict_proba(X_tr[va_idx])
            te_acc += m.predict_proba(X_te) / N_FOLDS
        else:
            oof[va_idx] = softmax(m.decision_function(X_tr[va_idx]))
            te_acc += softmax(m.decision_function(X_te)) / N_FOLDS
    oof_f1 = f1_score(y, np.argmax(oof, axis=1), average='macro')
    print(f"  {name}: OOF F1={oof_f1:.4f}")
    oof_probas[name] = oof
    test_probas[name] = te_acc

print(f"Level 1 done in {time.time()-t0:.0f}s")

In [ ]:
# =============================================================================
# Cell 5: Level 2 — Meta-Learner + Alpha Search for blend
# =============================================================================

model_names = list(oof_probas.keys())
X_meta_train = np.hstack([oof_probas[name] for name in model_names])
X_meta_test  = np.hstack([test_probas[name] for name in model_names])

meta_oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
meta_test_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
for fi, (tr_idx, va_idx) in enumerate(folds):
    meta_lr = LogisticRegression(class_weight='balanced', C=1.0, max_iter=5000,
                                 solver='lbfgs', multi_class='multinomial', random_state=42)
    meta_lr.fit(X_meta_train[tr_idx], y[tr_idx])
    meta_oof[va_idx] = meta_lr.predict_proba(X_meta_train[va_idx])
    meta_test_acc += meta_lr.predict_proba(X_meta_test) / N_FOLDS

meta_f1 = f1_score(y, np.argmax(meta_oof, axis=1), average='macro')
print(f"Meta-learner OOF: {meta_f1:.4f}")

# V12 baseline
oof_svc_ens = 0.25 * oof_probas['svc_A'] + 0.40 * oof_probas['svc_F'] + 0.35 * oof_probas['svc_F2']
oof_v12 = 0.70 * oof_svc_ens + 0.30 * oof_probas['lgb1']
v12_f1 = f1_score(y, np.argmax(oof_v12, axis=1), average='macro')
print(f"V12-style OOF: {v12_f1:.4f}")

# NOVO: Alpha search for meta+v12 blend
te_svc_ens = 0.25 * test_probas['svc_A'] + 0.40 * test_probas['svc_F'] + 0.35 * test_probas['svc_F2']
te_v12 = 0.70 * te_svc_ens + 0.30 * test_probas['lgb1']

best_alpha = None
best_blend_f1 = 0
print("\nAlpha search (meta * alpha + v12 * (1-alpha)):")
for alpha in np.arange(0.0, 1.05, 0.05):
    blend = alpha * meta_oof + (1 - alpha) * oof_v12
    bf1 = f1_score(y, np.argmax(blend, axis=1), average='macro')
    marker = " ***" if bf1 > best_blend_f1 else ""
    print(f"  alpha={alpha:.2f}: F1={bf1:.4f}{marker}")
    if bf1 > best_blend_f1:
        best_blend_f1 = bf1
        best_alpha = alpha

print(f"\nBest alpha={best_alpha:.2f}: F1={best_blend_f1:.4f}")

# Final blend
oof_blend = best_alpha * meta_oof + (1 - best_alpha) * oof_v12
te_blend = best_alpha * meta_test_acc + (1 - best_alpha) * te_v12

In [ ]:
# =============================================================================
# Cell 6: NOVO — 2-Pass Threshold Search + Multiple Orderings
# =============================================================================

def apply_thresholds_ordered(proba, thresholds, class_order):
    """Apply thresholds in a specific class order"""
    preds = np.argmax(proba, axis=1).copy()
    for cls in class_order:
        if cls in thresholds:
            mask = proba[:, cls] > thresholds[cls]
            # Don't override classes already set by higher-priority thresholds
            for higher_cls in class_order:
                if higher_cls == cls: break
                if higher_cls in thresholds:
                    mask = mask & (preds != higher_cls)
            preds[mask] = cls
    return preds

def greedy_threshold_search(proba, y_true, class_order, thr_range):
    """Greedy sequential threshold search for given class order"""
    best_thresholds = {}
    current_f1 = f1_score(y_true, np.argmax(proba, axis=1), average='macro')
    
    for cls in class_order:
        best_cls_f1 = current_f1
        best_cls_thr = None
        for thr in thr_range:
            trial = {**best_thresholds, cls: thr}
            trial_preds = apply_thresholds_ordered(proba, trial, class_order)
            trial_f1 = f1_score(y_true, trial_preds, average='macro')
            if trial_f1 > best_cls_f1:
                best_cls_f1 = trial_f1
                best_cls_thr = thr
        if best_cls_thr is not None:
            best_thresholds[cls] = best_cls_thr
            current_f1 = best_cls_f1
    
    return best_thresholds, current_f1

# Multiple orderings to escape local optima
orderings = [
    [6, 5, 4, 3, 1, 0],  # Original EXP-24 order (high to low risk)
    [0, 1, 3, 4, 5, 6],  # Low to high risk
    [5, 6, 4, 3, 0, 1],  # Most confused classes first
    [6, 4, 5, 3, 0, 1],  # Alternative
    [3, 5, 6, 4, 1, 0],  # Rare classes first
]

# PASS 1: Coarse search (step 0.02)
print("=== PASS 1: Coarse threshold search ===")
coarse_range = np.arange(0.05, 0.55, 0.02)
best_overall_f1 = 0
best_overall_thresholds = {}
best_overall_order = None

for ordering in orderings:
    thresholds, f1_val = greedy_threshold_search(oof_blend, y, ordering, coarse_range)
    marker = " <<<" if f1_val > best_overall_f1 else ""
    print(f"  Order {ordering}: F1={f1_val:.4f} thresholds={thresholds}{marker}")
    if f1_val > best_overall_f1:
        best_overall_f1 = f1_val
        best_overall_thresholds = thresholds
        best_overall_order = ordering

print(f"\nBest coarse: F1={best_overall_f1:.4f}, order={best_overall_order}")

# PASS 2: Fine search around best coarse thresholds (step 0.002)
print("\n=== PASS 2: Fine threshold refinement ===")
fine_thresholds = {}
current_f1 = f1_score(y, np.argmax(oof_blend, axis=1), average='macro')

for cls in best_overall_order:
    if cls in best_overall_thresholds:
        center = best_overall_thresholds[cls]
        fine_range = np.arange(max(0.01, center - 0.03), min(0.60, center + 0.03), 0.002)
    else:
        fine_range = np.arange(0.05, 0.55, 0.01)  # Full search for classes not found in coarse
    
    best_cls_f1 = current_f1
    best_cls_thr = None
    for thr in fine_range:
        trial = {**fine_thresholds, cls: thr}
        trial_preds = apply_thresholds_ordered(oof_blend, trial, best_overall_order)
        trial_f1 = f1_score(y, trial_preds, average='macro')
        if trial_f1 > best_cls_f1:
            best_cls_f1 = trial_f1
            best_cls_thr = thr
    if best_cls_thr is not None:
        fine_thresholds[cls] = best_cls_thr
        current_f1 = best_cls_f1
        print(f"  Class {cls}: threshold={best_cls_thr:.3f}, F1={best_cls_f1:.4f}")
    else:
        print(f"  Class {cls}: no improvement")

final_oof_preds = apply_thresholds_ordered(oof_blend, fine_thresholds, best_overall_order)
final_oof_f1 = f1_score(y, final_oof_preds, average='macro')
print(f'\nFinal OOF with refined thresholds: {final_oof_f1:.4f}')
print(f'Thresholds: {fine_thresholds}')
print(f'Order: {best_overall_order}')
print(classification_report(y, final_oof_preds, digits=4))

best_thresholds = fine_thresholds
threshold_order = best_overall_order

In [ ]:
# =============================================================================
# Cell 7: NOVO — Guardrails refinados + Submission
# =============================================================================

test_preds = apply_thresholds_ordered(te_blend, best_thresholds, threshold_order)
test['target'] = test_preds

def apply_clinical_guardrails_v2(row):
    text = str(row['report']).lower()
    pred = int(row['target'])
    
    # BI-RADS explicito — mas APENAS se aparece na seção de conclusão/impressão
    # (evita pegar BI-RADS mencionado no contexto clinico sem ser o diagnostico)
    conclusion_section = extract_conclusion(row['report'])
    if conclusion_section:
        birads_match = re.search(r'(?:bi-?rads|categoria)\s*:?\s*(\d)', conclusion_section)
        if birads_match:
            mentioned = int(birads_match.group(1))
            if 0 <= mentioned <= 6:
                return mentioned
    
    # Fallback: BI-RADS no texto completo (mas só se formato estruturado)
    birads_match = re.search(r'(?:bi-?rads|categoria)\s*:?\s*(\d)', text)
    if birads_match:
        mentioned = int(birads_match.group(1))
        if 0 <= mentioned <= 6:
            return mentioned
    
    # Malignidade confirmada
    if re.search(r'resultado de cine grau 3|carcinoma|\bcdis\b|neoplasia maligna', text):
        return 6
    
    # Morfologia altamente suspeita → pelo menos 5
    if re.search(r'n[oó]dulo\s+espiculad', text) and pred < 4:
        return 5
    if 'espiculad' in text and 'distor' in text and pred < 4:
        return 5
    
    # Calcificacao benigna downgrade
    if re.search(r'calcifica[cç][aã]o grosseira|calcifica[cç][aã]o vascular', text):
        if pred >= 4 and not re.search(r'espiculad|suspeit|malign|distor', text):
            return 2
    
    # NOVO: Exame negativo / mama normal → classe 1
    if re.search(r'exame negativo|mamas sem altera[cç][oõ]es|estudo mamogr[aá]fico normal|nada digno de nota', text):
        if pred >= 3 and not re.search(r'n[oó]dulo|calcifica|assimetria|distor|suspeit', text):
            return 1
    
    return pred

test['target'] = test.apply(apply_clinical_guardrails_v2, axis=1)

submission = test[['ID', 'target']].copy()
submission['target'] = submission['target'].astype(int)
submission.to_csv('submission.csv', index=False)

print('Prediction distribution:')
print(submission['target'].value_counts().sort_index())
print(f'\nSubmission saved! Shape: {submission.shape}')
print(submission)